# Advanced OCR — Colab runner (curated training-corpus builder)

> **Runtime: use CPU (none).** A GPU is NOT needed — the OXR-1.0 model runs on SoMark's
> servers; your runtime only slices PDFs (CPU) and makes HTTP calls. Switch via
> *Runtime → Change runtime type → CPU* and save your GPU quota.

This notebook is **only a runner**. All code lives in the GitHub repo
`greencube-dev-boop/advanced-ocr` (synced below with `git pull` — push updates to
GitHub and just rerun this notebook, never re-upload it). All data lives in your
Google Drive under **`OCR Engine/`**.

**Purpose: build a training corpus for a scientific LLM.** Every page of every PDF is
parsed by SoMarkAI/OXR-1.0 (tables → Markdown, formulas → LaTeX, chemical equations →
mhchem, chemical structures → SMILES, code blocks → fences), then a curation pass keeps
**body content only**: title/copyright pages, ToC pages, acknowledgements,
preface/foreword, references/bibliography, index, appendices, blank and figure-only
pages are dropped. Each book assembles into one **pure-content** `.md` file.

**Intake & lifecycle:** `raw/` is scanned recursively. Before processing, each PDF's
sha256 is checked against the ledger (duplicates are skipped and archived). Completed
books get their temp files pruned and their source PDF archived to `raw/_done/`; books
that keep failing are quarantined to `raw/_failed/`. The permanent ledger
(`config/books.json`) records sha256, DOI, ISBN, kept/dropped pages per book, and each
output file gets a sidecar `<book>.record.json`.

**No API keys are needed** (free OXR API). The GitHub repo is **public** — no token to
sync either.

**Rerunning this notebook resumes where it stopped.** `ocr_pipeline reset <book>` undoes
a book (move its PDF out of `raw/_done/` back into `raw/` first).

In [ ]:
# 1) Dependencies (idempotent)
!pip install -q PyMuPDF requests tqdm
print("deps ok")

In [ ]:
# 2) Mount Google Drive and set the OCR Engine root
import os
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/OCR Engine"
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("Drive root:", DRIVE_ROOT)
for sub in ("raw", "config", "refined", "temp"):
    os.makedirs(os.path.join(DRIVE_ROOT, sub), exist_ok=True)
print("raw/ is scanned recursively — drop PDFs into any subfolder of:",
      os.path.join(DRIVE_ROOT, "raw"))

In [ ]:
# 3) Sync the codebase from GitHub (clone once, then git pull on every run)
#    The repo is PUBLIC: no token needed. The GITHUB_TOKEN fallbacks below only
#    matter if the repo is ever made private again.
import os, shutil, subprocess, sys

REPO_URL = "https://github.com/greencube-dev-boop/advanced-ocr.git"
GITHUB_TOKEN = ""
WORKDIR = "/content/advanced-ocr"

try:
    from google.colab import userdata
    GITHUB_TOKEN = GITHUB_TOKEN or userdata.get("GITHUB_TOKEN")
    print("token source: Colab Secrets")
except Exception:
    print("token source:", "in-cell variable" if GITHUB_TOKEN else "NONE (anonymous clone — repo is public)")

url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL

if os.path.isdir(os.path.join(WORKDIR, ".git")):
    subprocess.run(["git", "-C", WORKDIR, "pull", "--ff-only", url], check=False)
else:
    shutil.rmtree(WORKDIR, ignore_errors=True)   # clear any partial clone
    r = subprocess.run(["git", "clone", url, WORKDIR])
    if r.returncode != 0:
        shutil.rmtree(WORKDIR, ignore_errors=True)
        raise SystemExit(
            f"git clone failed (exit {r.returncode}). If the repo is private, do ONE of:\n"
            "  a) Colab Secrets: add GITHUB_TOKEN = your GitHub PAT (recommended), then rerun this cell\n"
            "  b) paste your PAT into GITHUB_TOKEN = \"...\" in this cell, then rerun\n"
            "  c) make the repo public (the code contains no keys)")

os.environ["PYTHONPATH"] = os.path.join(WORKDIR, "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, os.path.join(WORKDIR, "src"))
import ocr_pipeline
print("pipeline", ocr_pipeline.__version__, "from", WORKDIR)

## About API keys and OXR modes

- **No OCR keys are required.** The free OXR API (`oxr.mode=free_api`, default) needs no sign-up.
  Limits per request: 10 MB / 20 pages — the pipeline builds chunks that respect both.
  The free API is queued and best-effort; *contingency* if it ever degrades: self-hosted OXR
  (needs a ≥24 GB BF16 GPU) or a paid SoMark key — only `oxr.mode`/`base_url`/`api_key` change.
- **Self-hosting OXR** (`oxr.mode=local`): run the [OXR services](https://github.com/SoMarkAI/OXR)
  yourself and point `oxr.local_base_url` at it (default `http://127.0.0.1:8000/v1`).

In [ ]:
# 4) Sanity check: deps, drive, OXR reachability (also lists every PDF found under raw/)
!python -m ocr_pipeline doctor --drive-root "{DRIVE_ROOT}"
!python -m ocr_pipeline init-config --drive-root "{DRIVE_ROOT}"

In [ ]:
# 5) Run / resume the pipeline.
#    Drop PDFs into Drive -> "OCR Engine/raw/" (any subfolders) first.
#    Finished pages are skipped automatically; interrupt anytime and rerun to resume.
!python -m ocr_pipeline run --drive-root "{DRIVE_ROOT}"

In [ ]:
# 6) Inspect the ledger
!python -m ocr_pipeline status --drive-root "{DRIVE_ROOT}"

## Outputs

- `Drive/OCR Engine/refined/<same relative path as raw>/<book>.md` — pure training text
- `<book>.record.json` next to it — provenance (sha256, DOI, ISBN, kept/dropped pages)
- `config/books.json` — the permanent ledger
- `raw/_done/`, `raw/_failed/` — archived source PDFs

## Tips

- Curation patterns (what counts as trash) live under `curation.*` in
  `Drive/OCR Engine/config/pipeline.json` — edit them there, no code changes needed.
- Reprocess a book: move its PDF from `raw/_done/` back to `raw/`, then
  `!python -m ocr_pipeline reset <book>.pdf --drive-root "{DRIVE_ROOT}"` and run again.
